In [ ]:
!pip install -q clearml

%env CLEARML_WEB_HOST=
%env CLEARML_API_HOST=
%env CLEARML_FILES_HOST=
%env CLEARML_API_ACCESS_KEY=
%env CLEARML_API_SECRET_KEY=

In [ ]:
import os
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from clearml import Task
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T

TRAIN_TASK_ID = ''

train_task = Task.get_task(task_id=TRAIN_TASK_ID)
CKPT_PATH = train_task.artifacts['best_weights'].get_local_copy()
THRESHOLDS_PATH = train_task.artifacts['best_thresholds_json'].get_local_copy()

class MLPHead(nn.Module):
    def __init__(self, in_features: int, hidden_dim: int = 256, dropout_p: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class OcclusionNet_V2(nn.Module):
    def __init__(self, num_classes: int = 7, hidden_dim: int = 256, dropout_p: float = 0.3):
        super().__init__()
        weights = models.EfficientNet_B3_Weights.DEFAULT
        self.backbone = models.efficientnet_b3(weights=weights)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.heads = nn.ModuleList([
            MLPHead(in_features=in_features, hidden_dim=hidden_dim, dropout_p=dropout_p)
            for _ in range(num_classes)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        logits = [head(features) for head in self.heads]
        return torch.cat(logits, dim=1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

VAL_ROOT = Path('/kaggle/input/datasets/saveliymazovatov/val-occ/val_samples')
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

CLASSES = [
    "DaytimeFlare",
    "Fog",
    "MotionBlur",
    "NighttimeFlare",
    "Raindrops",
    "Reflections",
    "Soil"
]

model = OcclusionNet_V2(num_classes=len(CLASSES), hidden_dim=256, dropout_p=0.3).to(device)

if 'CKPT_PATH' in globals() and os.path.exists(CKPT_PATH):
    state_dict = torch.load(CKPT_PATH, map_location=device)
    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]
    model.load_state_dict(state_dict, strict=True)

model.eval()

inference_transform = T.Compose([
    T.Resize((300, 300)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

samples = []
if VAL_ROOT.exists():
    for class_dir in sorted(VAL_ROOT.iterdir()):
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name
        if class_name not in CLASSES and class_name.lower() not in {'clean', 'clear'}:
            continue
        files = sorted(p for p in class_dir.iterdir() if p.suffix.lower() in IMG_EXTS)
        samples += [(p, class_name) for p in files]

all_probs = []
all_targets = []
num_classes = len(CLASSES)

with torch.no_grad():
    for path, true_class in samples:
        image = Image.open(path).convert('RGB')
        tensor = inference_transform(image).unsqueeze(0).to(device)
        
        with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            logits = model(tensor)
            probs = torch.sigmoid(logits)[0].cpu().numpy()
            
        target = np.zeros(num_classes, dtype=np.float32)
        if true_class in CLASSES:
            target[CLASSES.index(true_class)] = 1.0
            
        all_probs.append(probs)
        all_targets.append(target)

all_probs = np.array(all_probs)
all_targets = np.array(all_targets)

with open(THRESHOLDS_PATH, 'r') as f:
    saved_thresholds = json.load(f)

saved_thresholds = {k: float(v) for k, v in saved_thresholds.items()}

best_thresholds = {}
candidate_thresholds = np.arange(0.1, 0.9, 0.02)

for idx, name in enumerate(CLASSES):
    best_th = 0.5
    best_th_f1 = -1.0
    for th in candidate_thresholds:
        preds_th = (all_probs[:, idx] >= th).astype(int)
        score = f1_score(all_targets[:, idx], preds_th, average='binary', zero_division=0)
        if (score > best_th_f1) or (score == best_th_f1 and abs(th - 0.5) < abs(best_th - 0.5)):
            best_th_f1 = score
            best_th = th
    best_thresholds[name] = round(float(best_th), 2)

thresholds_05 = {name: 0.5 for name in CLASSES}

def compute_metrics(probs, targets, thresholds_map, class_names):
    th_list = [thresholds_map.get(c, 0.5) for c in class_names]
    preds = np.zeros_like(probs)
    for idx, th in enumerate(th_list):
        preds[:, idx] = (probs[:, idx] >= th).astype(float)
        
    report_data = []
    for idx, name in enumerate(class_names):
        p = precision_score(targets[:, idx], preds[:, idx], average='binary', zero_division=0)
        r = recall_score(targets[:, idx], preds[:, idx], average='binary', zero_division=0)
        f = f1_score(targets[:, idx], preds[:, idx], average='binary', zero_division=0)
        
        report_data.append({
            'Class': name,
            'Threshold': round(float(th_list[idx]), 2),
            'Precision': round(float(p), 4),
            'Recall': round(float(r), 4),
            'F1-Score': round(float(f), 4),
            'Support': int(targets[:, idx].sum())
        })
        
    clean_true = (targets.sum(axis=1) == 0).astype(int)
    clean_pred = (preds.sum(axis=1) == 0).astype(int)
    if clean_true.sum() > 0:
        p_clean = precision_score(clean_true, clean_pred, zero_division=0)
        r_clean = recall_score(clean_true, clean_pred, zero_division=0)
        f_clean = f1_score(clean_true, clean_pred, zero_division=0)
        
        report_data.append({
            'Class': 'Clean',
            'Threshold': '-',
            'Precision': round(float(p_clean), 4),
            'Recall': round(float(r_clean), 4),
            'F1-Score': round(float(f_clean), 4),
            'Support': int(clean_true.sum())
        })
        
    defect_rows = [d for d in report_data if d['Class'] != 'Clean']
    table_rows = list(report_data)
    table_rows.append({
        'Class': 'Macro (Defects Only)', 'Threshold': '-',
        'Precision': round(float(np.mean([d['Precision'] for d in defect_rows])), 4),
        'Recall': round(float(np.mean([d['Recall'] for d in defect_rows])), 4),
        'F1-Score': round(float(np.mean([d['F1-Score'] for d in defect_rows])), 4),
        'Support': int(sum([d['Support'] for d in defect_rows]))
    })
    table_rows.append({
        'Class': 'Macro (With Clean)', 'Threshold': '-',
        'Precision': round(float(np.mean([d['Precision'] for d in report_data])), 4),
        'Recall': round(float(np.mean([d['Recall'] for d in report_data])), 4),
        'F1-Score': round(float(np.mean([d['F1-Score'] for d in report_data])), 4),
        'Support': int(sum([d['Support'] for d in report_data]))
    })
    return pd.DataFrame(table_rows)

df_05 = compute_metrics(all_probs, all_targets, thresholds_05, CLASSES)
df_saved = compute_metrics(all_probs, all_targets, saved_thresholds, CLASSES)
df_tuned = compute_metrics(all_probs, all_targets, best_thresholds, CLASSES)

print("\nРезультаты со стандартным порогом 0.5:")
print(df_05.to_string(index=False))

print("\nРезультаты с порогами из файла (best_thresholds.json):")
print(df_saved.to_string(index=False))

print("\nРезультаты с подобранными порогами на текущей выборке:")
print(df_tuned.to_string(index=False))

ACTIVE_THRESHOLDS = saved_thresholds

@torch.no_grad()
def predict_and_show(image_path, true_class, model, transform, classes, device, thresholds_map, show_plot=True):
    image = Image.open(image_path).convert('RGB')
    tensor = transform(image).unsqueeze(0).to(device)

    with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
        logits = model(tensor)
        probs = torch.sigmoid(logits)[0].cpu().numpy()

    detected = [
        (c, float(p)) for c, p in zip(classes, probs) 
        if p >= thresholds_map.get(c, 0.5)
    ]
    detected.sort(key=lambda x: -x[1])
    pred_names = {c for c, _ in detected}
    
    if true_class.lower() in {'clean', 'clear'}:
        correct = (len(detected) == 0)
    else:
        correct = (true_class in pred_names) if true_class in classes else None

    if show_plot:
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(image)
        ax.axis('off')

        pred_line = '  |  '.join(f'{c}: {p:.2f}' for c, p in detected) if detected else 'Clean — окклюзий не обнаружено'
        mark = '' if correct is None else (' ✓' if correct else ' ✗')
        ax.set_title(f'{image_path.name}  [истина: {true_class}{mark}]\n{pred_line}', fontsize=10)

        full_line = '   '.join(f'{c}={p:.2f} (th:{thresholds_map.get(c, 0.5):.2f})' for c, p in zip(classes, probs))
        fig.text(0.5, 0.02, full_line, ha='center', fontsize=8, color='gray')

        plt.tight_layout()
        plt.show()

    return detected, correct


Результаты со стандартным порогом 0.5:
               Class Threshold  Precision  Recall  F1-Score  Support
        DaytimeFlare       0.5     1.0000  0.9000    0.9474       10
                 Fog       0.5     0.9091  1.0000    0.9524       10
          MotionBlur       0.5     1.0000  0.8000    0.8889       10
      NighttimeFlare       0.5     0.9091  1.0000    0.9524       10
           Raindrops       0.5     1.0000  1.0000    1.0000       10
         Reflections       0.5     0.9091  1.0000    0.9524       10
                Soil       0.5     1.0000  1.0000    1.0000       10
Macro (Defects Only)         -     0.9610  0.9571    0.9562       70
  Macro (With Clean)         -     0.9610  0.9571    0.9562       70

Результаты с порогами из файла (best_thresholds.json):
               Class Threshold  Precision  Recall  F1-Score  Support
        DaytimeFlare      0.54     1.0000  0.9000    0.9474       10
                 Fog      0.64     0.9091  1.0000    0.9524       10
       

In [8]:
total_params = sum(p.numel() for p in model.parameters())

print(f"Всего параметров:       {total_params:,}")
print(f"Размер в памяти (FP32): ~{total_params * 4 / (1024 ** 2):.2f} MB")

Всего параметров:       13,455,919
Размер в памяти (FP32): ~51.33 MB


In [ ]:
print(f"\nЗапуск отрисовки картинок с порогами: {ACTIVE_THRESHOLDS}")
for path, true_class in samples:
    predict_and_show(
        path, true_class, model, inference_transform, CLASSES, device, ACTIVE_THRESHOLDS, show_plot=True
    )